This notebook is a component-level ablation of RDS Fusion. It keeps the same Qwen2.5-3B-Instruct generation, RoBERTa classifier, confidence-score extraction, entropy-gated fusion, and symbolic irony prior as RDS Fusion, but completely bypasses the Local Guardian and all post-generation pruning. Qwen generation is strictly capped at max_new_tokens=120.

In [1]:
!pip install -q transformers datasets accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.1 MB/s eta 0:00:00:00:0100:01


We load the TweetEval dataset, the RoBERTa classifier, and the Qwen2.5 generator using the same model configuration as RDS Fusion.

In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Dataset (TweetEval - Irony)
print("Loading Dataset...")
dataset = load_dataset("tweet_eval", "irony")
test_data = dataset['test']

# 2. Load Evaluation Classifier (RoBERTa)
print("Loading RoBERTa Classifier...")
roberta_name = "cardiffnlp/twitter-roberta-base-irony"
rob_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
rob_model = AutoModelForSequenceClassification.from_pretrained(roberta_name).to(device)

# 3. Define the 4-bit Quantization Configuration
print("Configuring 4-bit Quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # Speeds up computation on T4
)

# 4. Load Reasoning Generator (Qwen 3B with updated quantization)
print("Loading Qwen2.5-3B...")
qwen_name = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    device_map="auto",
    quantization_config=bnb_config
)

Loading Dataset...


README.md: 0.00B [00:00, ?B/s]

irony/train-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

irony/test-00000-of-00001.parquet:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

irony/validation-00000-of-00001.parquet:   0%|          | 0.00/61.1k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

This cell computes the same generation entropy used by the entropy-gated adaptive alpha in RDS Fusion. No token pruning or Local Guardian operation is performed.

In [3]:
import torch
import torch.nn.functional as F
import math

def calculate_dynamic_gamma(logits, min_gamma=0.1, max_gamma=0.9, k=1, mu=0.5):
    """
    Calculates the reasoning entropy and maps it to a dynamic compression ratio.
    """
    # Convert logits to probabilities
    probs = F.softmax(logits, dim=-1)

    # Epsilon (1e-9) to prevent log(0) yielding -inf and breaking the sum into a nan
    epsilon = 1e-9
    entropy_tensor = -torch.sum(probs * torch.log(probs + epsilon), dim=-1)
    entropy = entropy_tensor.mean().item()

    # Fallback catch just in case a nan slips through due to precision limits
    if math.isnan(entropy):
        entropy = 0.0

    # USING math.exp instead of np.exp since entropy is a standard float here
    gamma_dyn = min_gamma + (max_gamma - min_gamma) / (1 + math.exp(-k * (entropy - mu)))

    return gamma_dyn, entropy

The following helper functions are kept unchanged from RDS Fusion wherever they affect scoring and fusion. In particular, the confidence extraction and two-pass fallback are identical. The compression function and Local Guardian are omitted.

In [4]:
import torch
import torch.nn.functional as F
import nltk, re
import pandas as pd
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)


# ═════════════════════════════════════════════════════════════════════════════
# PROMPTS  — continuous confidence scoring with explicit hashtag guidance
# ═════════════════════════════════════════════════════════════════════════════

STRUCTURED_PROMPT = """\
You are an expert linguist specializing in detecting irony and sarcasm in social media text.

DEFINITION: A tweet is IRONIC if there is a contrast between its literal meaning and its \
intended meaning, or if the author says the opposite of what they actually mean (often to \
mock, criticize, or be humorous). A tweet is NON-IRONIC if it is a sincere, literal statement.

KEY SIGNALS TO CHECK:
  - Does the literal meaning contradict the real-world situation?
  - Is there an exaggerated, over-the-top positive/negative tone?
  - Are there hashtags like #not, #sarcasm, #irony, #obviously that signal ironic intent?
  - Does the tweet mock or criticize something by pretending to praise it?
  - Would a reasonable reader take this at face value, or detect a hidden meaning?
  - Are there elongated words like "Loooove" or "Soooo" used sarcastically?

CONFIDENCE SCALE:
  0.0 = Absolutely certain NON-IRONIC (sincere, literal, no ambiguity at all)
  0.1 = Very likely non-ironic, tiny residual doubt
  0.3 = Probably non-ironic, some mixed signals present
  0.5 = Completely uncertain — could genuinely be either
  0.7 = Probably ironic, some mixed signals present
  0.9 = Very likely ironic, tiny residual doubt
  1.0 = Absolutely certain IRONIC (clear sarcasm/irony, no ambiguity at all)

EXAMPLES:
Tweet: "Oh great, another Monday. Just what I needed."
Reasoning: "Just what I needed" is exaggeratedly positive about something universally \
disliked. Classic sarcasm with no ambiguity.
Score: 0.95

Tweet: "Happy birthday to my best friend! Hope your day is amazing."
Reasoning: Sincere, literal birthday wish. No hidden meaning, no contrast, tone \
matches content perfectly.
Score: 0.05

Tweet: "Wow, love how my flight got cancelled on the day of my interview. Truly blessed."
Reasoning: "Truly blessed" after describing a disaster is a clear ironic inversion. \
Very high confidence.
Score: 0.92

Tweet: "This weather is something else today."
Reasoning: Ambiguous — could be genuine admiration or sarcastic complaint depending \
on context not available in the tweet alone.
Score: 0.50

Now analyze the following tweet using the same reasoning process.

Tweet: '{tweet}'

Think step by step through the KEY SIGNALS above. End your response with EXACTLY:
Score: X.XX
(a number between 0.00 and 1.00, two decimal places)"""


FORCED_VERDICT_PROMPT = """\
You analyzed a tweet and wrote this reasoning:
---
{cot_text}
---

Based solely on your own analysis above, rate your confidence that the tweet is ironic.

Output ONLY a single line in this exact format, nothing else:
Score: X.XX
(0.00 = definitely non-ironic, 0.50 = uncertain, 1.00 = definitely ironic)"""


# ═════════════════════════════════════════════════════════════════════════════
# EXPLICIT IRONY SIGNAL DETECTOR
# Catches author-labeled irony signals that both RoBERTa and CoT systematically
# miss: #not, #sarcasm, #irony, emoji contrast, letter elongation.
# ═════════════════════════════════════════════════════════════════════════════

_STRONG_IRONY_RE = re.compile(
    r"#(not|sarcasm|sarcastic|irony|ironic|jk|justkidding|kms|killme|"
    r"killusslow|obviously|surenot|yeahright|fml|eyeroll)\b",
    re.IGNORECASE
)
_WEAK_IRONY_RE = re.compile(
    r"#(humor|lol|smh|seriously|really|wow|sure|totally|great|wonderful)\b",
    re.IGNORECASE
)
# Positive emoji followed immediately by negative emoji → contrast = sarcasm
_EMOJI_CONTRAST_RE = re.compile(
    r"[😊😄😀👍❤️🙂😁]\s{0,3}[|,\s]*\s{0,3}[😒😤😡😞😑🙄😔😢]"
)
# Letter elongation: "Loooovvveee", "Soooo" (3+ repeated chars)
_ELONGATION_RE = re.compile(r"([a-zA-Z])\1{2,}")

def detect_explicit_irony_signal(tweet_text):
    """
    Scans tweet text for explicit, author-provided irony/sarcasm markers.

    Returns a float in [0.0, 0.88]:
      0.88  → Strong signal (#not, #sarcasm, #irony etc.) — near-certain irony
      0–0.70 → Weak composite signal (soft hashtags + emoji contrast + elongation)
      0.0   → No explicit signal detected

    Design note: Even at 0.88, the blending formula (60/40) means a COMBINED
    very-wrong RoBERTa+CoT can still override, preventing false positives.
    """
    if _STRONG_IRONY_RE.search(tweet_text):
        return 0.88

    score = 0.0
    if _WEAK_IRONY_RE.search(tweet_text):
        score += 0.15
    if _EMOJI_CONTRAST_RE.search(tweet_text):
        score += 0.25
    if _ELONGATION_RE.search(tweet_text):
        score += 0.12
    return min(score, 0.70)


# ═════════════════════════════════════════════════════════════════════════════
# CONFIDENCE SCORE EXTRACTION  (unchanged from v4)
# ═════════════════════════════════════════════════════════════════════════════

def extract_cot_confidence(cot_text):
    """
    Parses Qwen's continuous confidence score from its CoT output.
    Returns float in [0.0, 1.0].

    Priority order:
      P1. Explicit "Score: X.XX" line
      P2. Legacy "Verdict: IRONIC/NON-IRONIC" line (backward compat)
      P3. Negation-aware weighted scan on last 3 sentences → mapped to [0.15, 0.85]
      P4. Full-text fallback → returns 0.5
    """
    lower = cot_text.lower().strip()

    # P1: Score line
    score_match = re.search(
        r"score\s*[:\-]\s*([01](?:\.\d{1,2})?|0?\.\d{1,2})",
        lower
    )
    if score_match:
        return max(0.0, min(1.0, float(score_match.group(1))))

    # P2: Legacy verdict line
    verdict_match = re.search(r"verdict\s*[:\-]\s*(.{1,40})", lower)
    if verdict_match:
        v = verdict_match.group(1).strip()
        if re.search(r"\b(non[- ]?ironic|not\s+ironic|not\s+sarcastic|"
                     r"sincere|literal|no\s+irony|no\s+sarcasm)\b", v):
            return 0.1
        if re.search(r"\b(ironic|sarcastic|sarcasm|irony)\b", v):
            return 0.9
        if re.search(r"^\s*(yes|correct|true|indeed)\s*$", v):
            return 0.9
        if re.search(r"^\s*(no|false|incorrect)\s*$", v):
            return 0.1

    # P3: Negation-aware scan on last 3 sentences
    sentences  = re.split(r"(?<=[.!?])\s+", cot_text.strip())
    window     = sentences[-3:] if len(sentences) >= 3 else sentences
    conclusion = " ".join(window).lower()

    NEGATED_IRONY  = re.compile(
        r"\b(not|no|isn'?t|aren'?t|doesn'?t|without|lacks?|free\s+from)\b"
        r".{0,25}\b(iron(?:ic|y)|sarcas(?:m|tic))\b"
    )
    PLAIN_IRONY    = re.compile(r"\b(iron(?:ic|y)|sarcas(?:m|tic))\b")
    CONCLUDE_WORDS = re.compile(
        r"\b(therefore|thus|hence|so|conclude|conclusion|overall|in\s+summary|"
        r"clearly|obviously|evident|appears?|seems?|suggest|indicate)\b"
    )
    DIRECT_IRONIC = re.compile(
        r"\b(this|the)\s+(tweet|text|statement|post)\s+(is|appears?|seems?)\s+"
        r"(?:to\s+be\s+)?(ironic|sarcastic)\b"
    )
    DIRECT_NONIRONIC = re.compile(
        r"\b(this|the)\s+(tweet|text|statement|post)\s+(is|appears?|seems?)\s+"
        r"(?:to\s+be\s+)?(?:not\s+|non[- ]?)(ironic|sarcastic)\b"
        r"|\b(genuine|sincere|literal|straightforward|earnest)\b"
    )

    ironic_score = nonironic_score = 0
    negated_spans = [(m.start(), m.end()) for m in NEGATED_IRONY.finditer(conclusion)]
    for s, e in negated_spans:
        ctx = conclusion[max(0,s-50):s] + conclusion[e:e+50]
        nonironic_score += 3 if CONCLUDE_WORDS.search(ctx) else 2
    for m in PLAIN_IRONY.finditer(conclusion):
        if any(s <= m.start() <= e for s, e in negated_spans):
            continue
        ctx = conclusion[max(0, m.start()-50):m.start()] + conclusion[m.end():m.end()+50]
        ironic_score += 3 if CONCLUDE_WORDS.search(ctx) else 2
    ironic_score    += len(DIRECT_IRONIC.findall(conclusion))    * 2
    nonironic_score += len(DIRECT_NONIRONIC.findall(conclusion)) * 2

    total = ironic_score + nonironic_score
    if total > 0:
        return 0.15 + (ironic_score / total) * 0.70

    # P4: Full-text fallback
    full_negated = len(NEGATED_IRONY.findall(lower))
    full_plain   = len(PLAIN_IRONY.findall(lower)) - full_negated
    total_full   = full_plain + full_negated
    if total_full > 0:
        return 0.15 + (max(0, full_plain) / total_full) * 0.70

    return 0.5


# ═════════════════════════════════════════════════════════════════════════════
# TWO-PASS FALLBACK
# ═════════════════════════════════════════════════════════════════════════════

def get_forced_score(cot_text, qwen_model, qwen_tokenizer, device):
    """Pass 2: triggered when Pass 1 returns exactly 0.5. Max 10 new tokens."""
    prompt  = FORCED_VERDICT_PROMPT.format(cot_text=cot_text[:800])
    inputs  = qwen_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(**inputs, max_new_tokens=10, do_sample=False)
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response      = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True)
    return extract_cot_confidence(response)


def get_cot_signal_with_fallback(cot_text, qwen_model, qwen_tokenizer, device):
    """Returns continuous float in [0.0, 1.0]. Runs Pass 2 only if Pass 1 gives 0.5."""
    score = extract_cot_confidence(cot_text)
    if score == 0.5:
        score = get_forced_score(cot_text, qwen_model, qwen_tokenizer, device)
    return score


# ═════════════════════════════════════════════════════════════════════════════
# ROBERTA CLASSIFIER  (unchanged)
# ═════════════════════════════════════════════════════════════════════════════

def classify_tweet(tweet_text, rob_tokenizer, rob_model, device):
    """Classify the raw tweet with RoBERTa. Returns (predicted_class, logits)."""
    inputs = rob_tokenizer(tweet_text, return_tensors="pt",
                           truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = rob_model(**inputs)
    logits     = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()
    return prediction, logits


# ═════════════════════════════════════════════════════════════════════════════
# ENTROPY-GATED ADAPTIVE ALPHA
# Changes: alpha_low 0.55→0.50, alpha_high 0.85→0.88
# Risk of tweet-10 regression mitigated by CoT skepticism applied before this.
# ═════════════════════════════════════════════════════════════════════════════

def entropy_to_alpha(entropy,
                     low_thresh=0.2,  high_thresh=0.5,
                     alpha_low=0.50,  alpha_high=0.88):
    """
    Maps reasoning entropy to RoBERTa weight (alpha).
      Low entropy  → Qwen was confident → trust CoT more → lower alpha (0.50)
      High entropy → Qwen was uncertain → trust RoBERTa more → higher alpha (0.88)
    """
    if entropy <= low_thresh:
        return alpha_low
    elif entropy >= high_thresh:
        return alpha_high
    t = (entropy - low_thresh) / (high_thresh - low_thresh)
    return alpha_low + t * (alpha_high - alpha_low)


# ═════════════════════════════════════════════════════════════════════════════
# FUSED CLASSIFICATION  v5
# New in this version vs v4:
#   1. CoT extreme score skepticism — pulls p_cot toward 0.5 when Qwen's
#      generation entropy contradicts its own claimed confidence.
#      This also mitigates regression risk from the wider alpha range.
#   2. Hashtag prior injection — explicit irony signals from tweet text.
#   3. Conflict override softened: threshold 0.80→0.85, alpha 0.90→0.85.
#      (Not removed — still valuable in extreme disagreement cases like tweet 55.)
# ═════════════════════════════════════════════════════════════════════════════

def fused_classify_v2(tweet_text, cot_text, entropy,
                      rob_tokenizer, rob_model,
                      qwen_model, qwen_tokenizer, device):
    """
    Entropy-gated fusion with hashtag injection and CoT skepticism.

    Full pipeline:
      1. RoBERTa → p_roberta
      2. Two-pass Qwen extraction → p_cot  (continuous [0,1])
      3. CoT extreme score skepticism: shrink toward 0.5 when entropy
         contradicts claimed confidence (prevents overconfident wrong CoT)
      4. Gentle clamping to [0.05, 0.95]
      5. entropy_to_alpha → base alpha
      6. Softened conflict override (threshold 0.85, override alpha 0.85)
      7. Standard fusion: p_fused = alpha*p_roberta + (1-alpha)*p_cot
      8. Hashtag prior injection (if detected, overrides step 7 result)

    Returns: (prediction, p_roberta, p_cot, alpha, p_fused, hashtag_prior)
    """
    # ── 1. RoBERTa ────────────────────────────────────────────────────────────
    _, tweet_logits = classify_tweet(tweet_text, rob_tokenizer, rob_model, device)
    tweet_probs = F.softmax(tweet_logits, dim=-1).squeeze().cpu()
    p_roberta   = tweet_probs[1].item()

    # ── 2. Qwen confidence score (two-pass) ───────────────────────────────────
    p_cot = get_cot_signal_with_fallback(cot_text, qwen_model, qwen_tokenizer, device)

    # ── 3. CoT extreme score skepticism ───────────────────────────────────────
    # When Qwen's generation entropy is moderate/high but its verdict is extreme
    # (very close to 0 or 1), it is likely overconfident. Shrink toward 0.5.
    # This prevents a wrong extreme CoT score from dominating the fusion.
    # Verified safe: in low-RoBERTa correct cases, fused still stays below 0.5.
    SKEPTICISM_ENTROPY_THRESH = 0.25   # entropy above this = non-trivial uncertainty
    SKEPTICISM_COT_THRESH     = 0.15   # p_cot this extreme is suspect under uncertainty
    if entropy > SKEPTICISM_ENTROPY_THRESH:
        if p_cot <= SKEPTICISM_COT_THRESH or p_cot >= (1 - SKEPTICISM_COT_THRESH):
            p_cot = 0.5 + (p_cot - 0.5) * 0.5   # halve distance from 0.5
            # e.g. 0.10 → 0.30,  0.95 → 0.725

    # ── 4. Gentle clamping ────────────────────────────────────────────────────
    p_cot = max(0.05, min(0.95, p_cot))

    # ── 5. Entropy-adaptive alpha ─────────────────────────────────────────────
    alpha = entropy_to_alpha(entropy)

    # ── 6. Softened conflict override ────────────────────────────────────────
    # Raised threshold (0.80→0.85) and lowered override alpha (0.90→0.85).
    # NOTE: CoT skepticism in step 3 already shrinks extreme p_cot values,
    # so this override fires less often than before — only for true extremes.
    if abs(p_roberta - p_cot) > 0.85:
        alpha = 0.85

    # ── 7. Standard fusion ────────────────────────────────────────────────────
    p_fused = alpha * p_roberta + (1 - alpha) * p_cot

    # ── 8. Hashtag prior injection ────────────────────────────────────────────
    hashtag_prior = detect_explicit_irony_signal(tweet_text)
    if hashtag_prior >= 0.85:
        # Strong signal: author explicitly labeled the tweet as ironic.
        # Drop alpha to give CoT more say, then blend hashtag prior at 40%.
        # Safety: even if both models are wrong (both say 0), p_fused = 0.352
        # which is still < 0.5, so this cannot force a wrong ironic prediction
        # when both models are confident the tweet is non-ironic.
        alpha_strong = min(alpha, 0.30)
        p_fused_base = alpha_strong * p_roberta + (1 - alpha_strong) * p_cot
        p_fused = 0.60 * p_fused_base + 0.40 * hashtag_prior
        alpha   = alpha_strong   # report the effective alpha used

    elif hashtag_prior > 0.0:
        # Weak signal: moderate blend
        p_fused = 0.75 * p_fused + 0.25 * hashtag_prior

    # ── 9. Decision ───────────────────────────────────────────────────────────
    prediction = int(p_fused >= 0.5)

    return prediction, p_roberta, p_cot, alpha, p_fused, hashtag_prior


# ═════════════════════════════════════════════════════════════════════════════

print("✅ Helper functions loaded.")
print("   • detect_explicit_irony_signal   — same symbolic prior as RDS Fusion")
print("   • extract_cot_confidence         — same continuous score parser")
print("   • get_cot_signal_with_fallback   — same two-pass pipeline")
print("   • classify_tweet                 — same RoBERTa classifier")
print("   • entropy_to_alpha               — same adaptive alpha")
print("   • fused_classify_v2              — same RDS Fusion downstream formulation")
print("   • Local Guardian / post-generation pruning — NOT USED")


✅ Helper functions loaded.
   • detect_explicit_irony_signal   — same symbolic prior as RDS Fusion
   • extract_cot_confidence         — same continuous score parser
   • get_cot_signal_with_fallback   — same two-pass pipeline
   • classify_tweet                 — same RoBERTa classifier
   • entropy_to_alpha               — same adaptive alpha
   • fused_classify_v2              — same RDS Fusion downstream formulation
   • Local Guardian / post-generation pruning — NOT USED


For the ablation, Qwen generates the same structured reasoning with a strict max_new_tokens=120 ceiling. The raw generated trajectory is passed directly to the unchanged RDS score-extraction and fusion pipeline. No Local Guardian, gradient-sensitivity calculation, dynamic token pruning, or compression is used.

In [7]:

def generate_cot_no_pruning(tweet, tweet_num=""):
    num_str = f" {tweet_num}" if tweet_num else ""
    print(f"\n--- Processing Tweet{num_str}: {tweet[:60]}... ---")

    prompt = STRUCTURED_PROMPT.format(tweet=tweet)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=120,
            return_dict_in_generate=True,
            output_scores=True
        )

    generated_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]
    cot_text = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Entropy is retained because it is an unchanged input to the adaptive
    # alpha fusion mechanism. It is not used for token pruning here.
    logits_stack = torch.stack(outputs.scores, dim=1).squeeze(0)
    gamma_dyn, entropy = calculate_dynamic_gamma(logits_stack)

    print(f"  CoT Length : {len(cot_text.split())} words")
    print(f"  Entropy    : {entropy:.3f}")

    return {
        "tweet": tweet,
        "cot_text": cot_text,
        "entropy": entropy,
        "gamma_dyn": gamma_dyn
    }


# Direct component ablation: first 50 tweets of the strictly held-out test set.
NUM_SAMPLES = len(test_data)

results_list = []
correct_predictions = 0

print(f"\nStarting NO-PRUNING RDS Ablation | N={NUM_SAMPLES}")
print("Generation: max_new_tokens=120 (strict ceiling)")
print("Local Guardian: DISABLED")
print("Post-generation pruning: DISABLED")
print("Score extraction: IDENTICAL TO RDS Fusion\n")

for i in range(50, NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label = test_data[i]["label"]

    result_dict = generate_cot_no_pruning(sample_tweet, tweet_num=i + 1)

    # Raw generated CoT goes directly into the unchanged downstream
    # score-extraction and fusion mechanism.
    prediction, p_roberta, cot_signal, alpha, p_fused, hashtag_prior = fused_classify_v2(
        sample_tweet,
        result_dict["cot_text"],
        result_dict["entropy"],
        rob_tokenizer, rob_model,
        qwen_model, qwen_tokenizer,
        device
    )

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    total_so_far = i - 49
    running_acc = (correct_predictions / total_so_far) * 100
    htag_str = f" | HTag={hashtag_prior:.2f}" if hashtag_prior > 0.0 else ""

    print(
        f"  RoBERTa={p_roberta:.3f} | CoT={cot_signal:.2f} | α={alpha:.3f}"
        f"{htag_str} | Fused={p_fused:.3f} | Pred={prediction} | True={true_label}"
        f" | {'✅' if is_correct else '❌'}"
        f"  [{correct_predictions}/{total_so_far} = {running_acc:.1f}%]"
    )

    orig_len = len(result_dict["cot_text"].split())

    results_list.append({
        "Tweet": sample_tweet[:35] + "...",
        "True": true_label,
        "Pred": prediction,
        "✓?": "✅" if is_correct else "❌",
        "RoBERTa P(ironic)": round(p_roberta, 3),
        "CoT Score": round(cot_signal, 3),
        "HTag Prior": round(hashtag_prior, 3),
        "α (adaptive)": round(alpha, 3),
        "Fused P(ironic)": round(p_fused, 3),
        "Entropy": round(result_dict["entropy"], 3),
        "Orig Len": orig_len,
    })

accuracy_pct = (correct_predictions / NUM_SAMPLES) * 100
htag_triggered = sum(1 for r in results_list if r["HTag Prior"] > 0.0)
htag_strong = sum(1 for r in results_list if r["HTag Prior"] >= 0.85)

print(f"\n{'=' * 64}")
print(
    f"NO-PRUNING RDS ABLATION ACCURACY : "
    f"{accuracy_pct:.1f}% ({correct_predictions}/{NUM_SAMPLES})"
)
print(f"🏷️ Hashtag detector fired         : {htag_triggered} tweets ({htag_strong} strong signals)")
print(f"✂️ Local Guardian / pruning       : DISABLED")
print(f"{'=' * 64}")

df = pd.DataFrame(results_list)
display(df)



Starting NO-PRUNING RDS Ablation | N=784
Generation: max_new_tokens=120 (strict ceiling)
Local Guardian: DISABLED
Post-generation pruning: DISABLED
Score extraction: IDENTICAL TO RDS Fusion


--- Processing Tweet 51: Rangers league game with Alloa moved because of the Petrofac... ---
  CoT Length : 85 words
  Entropy    : 0.256
  RoBERTa=0.565 | CoT=0.33 | α=0.300 | HTag=0.88 | Fused=0.590 | Pred=1 | True=1 | ✅  [1/1 = 100.0%]

--- Processing Tweet 52: How to Find a Life Coach (& the questions you need to ask be... ---
  CoT Length : 77 words
  Entropy    : 0.339
  RoBERTa=0.033 | CoT=0.68 | α=0.676 | Fused=0.241 | Pred=0 | True=0 | ✅  [2/2 = 100.0%]

--- Processing Tweet 53: I wonder what was the holiday rituals for true Africans... ---
  CoT Length : 78 words
  Entropy    : 0.360
  RoBERTa=0.209 | CoT=0.33 | α=0.702 | Fused=0.243 | Pred=0 | True=0 | ✅  [3/3 = 100.0%]

--- Processing Tweet 54: Been at the ER now since 10... #yay #not... ---
  CoT Length : 81 words
  Entropy    : 0.35

,Tweet,True,Pred,✓?,RoBERTa P(ironic),CoT Score,HTag Prior,α (adaptive),Fused P(ironic),Entropy,Orig Len
0,Rangers league game with Alloa move...,1,1,✅,0.565,0.325,0.88,0.300,0.590,0.256,85
1,How to Find a Life Coach (& the que...,0,0,✅,0.033,0.675,0.00,0.676,0.241,0.339,77
2,I wonder what was the holiday ritua...,0,0,✅,0.209,0.325,0.00,0.702,0.243,0.360,78
3,Been at the ER now since 10... #yay...,1,1,✅,0.344,0.675,0.88,0.300,0.697,0.352,81
4,@user Suhs been clean for 2 years a...,0,1,❌,0.041,0.675,0.88,0.300,0.643,0.363,80
...,...,...,...,...,...,...,...,...,...,...,...
729,"If you drag yesterday into today, y...",0,0,✅,0.027,0.250,0.00,0.567,0.123,0.253,87
730,Congrats to my fav @user & her team...,0,0,✅,0.082,0.250,0.00,0.627,0.145,0.301,76
731,@user Jessica sheds tears at her fa...,0,0,✅,0.326,0.675,0.00,0.622,0.458,0.296,78
732,#Irony: al jazeera is pro Anti - #G...,1,1,✅,0.796,0.500,0.88,0.300,0.705,0.392,91


Optional: compute the classification metrics for the ablation run.

In [8]:

from sklearn.metrics import accuracy_score, f1_score

y_true = [r["True"] for r in results_list]
y_pred = [r["Pred"] for r in results_list]

print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1  : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Ironic F1 : {f1_score(y_true, y_pred, pos_label=1):.4f}")


Accuracy  : 0.7548
Macro F1  : 0.7547
Ironic F1 : 0.7587


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = [r["True"] for r in results_list]
y_pred = [r["Pred"] for r in results_list]

ironic_precision = precision_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

ironic_recall = recall_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

non_ironic_f1 = f1_score(
    y_true, y_pred, pos_label=0, zero_division=0
)

print(f"Ironic Precision : {ironic_precision:.4f}")
print(f"Ironic Recall    : {ironic_recall:.4f}")
print(f"Non-Ironic F1    : {non_ironic_f1:.4f}")

Ironic Precision : 0.6193
Ironic Recall    : 0.9792
Non-Ironic F1    : 0.7507
